# EXACT 2026 Type-2 Physics GRPO RL Notebook

This notebook trains a **single Type-2 physics Qwen model** with GRPO-style reinforcement learning on the new synthetic RL datasets.

Design choices implemented here:

- Warm-start from the merged Option-B SFT model, defaulting to `not-a-real-ai-guy/qwen2.5-type2-option-b-modes-merged`.
- Load only the `_rl` Hugging Face datasets:
  - `not-a-real-ai-guy/exact_2026_type2_generator_rl`
  - `not-a-real-ai-guy/exact_2026_type2_explainer_rl`
- Merge generator/explainer rows by `SYN...` problem ID.
- Train GRPO only on `TASK_MODE=generate_code` completions.
- Use the paired explainer row and stored runner metadata only as reward context.
- Reuse the Type-2 safe Python/SymPy sandbox pattern and unit-aware answer checks.
- Keep A100-friendly defaults: BF16, `flash_attention_2` when available, left padding for GRPO generation.

> The model is **not** trained to generate code and explanation in the same GRPO completion. The reward loop optimizes the code-generation step because it causally determines downstream sandbox correctness and answer quality.


In [1]:
# ========================================================
# STEP 0: Install / upgrade dependencies with uv
# ========================================================
# Run this once per fresh runtime. Restart the kernel after this cell.
#
# IMPORTANT: this cell uses uv against sys.executable, so packages are changed
# in the exact Python environment used by this Jupyter kernel.
#
# This is a text-only Qwen/GRPO notebook. We intentionally remove torchvision.
# Some cloud images ship a torchvision wheel that does not match torch/CUDA and
# can break transformers/peft imports with:
#   RuntimeError: operator torchvision::nms does not exist

import sys, subprocess, importlib.util, os, shutil

print("Jupyter Python:", sys.executable)

# Ensure uv is available in this same Python environment.
# This small bootstrap uses pip only to install uv itself when uv is missing.
try:
    subprocess.check_call([sys.executable, "-m", "uv", "--version"])
except Exception:
    print("uv not found in this kernel env; installing uv first...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
    subprocess.check_call([sys.executable, "-m", "uv", "--version"])

uv = [sys.executable, "-m", "uv"]

pkgs = [
    "trl", "transformers", "accelerate", "peft", "datasets",
    "huggingface_hub", "sympy", "rapidfuzz", "safetensors"
]

# Install into the current Jupyter kernel Python, not a separate uv venv.
subprocess.check_call([
    *uv, "pip", "install",
    "--python", sys.executable,
    "-U",
    *pkgs,
])

# Remove torchvision from this exact environment. It is not needed for text-only RL.
# If it is not installed, uv will return a non-zero code on some versions; do not fail.
subprocess.call([
    *uv, "pip", "uninstall",
    "--python", sys.executable,
    "torchvision",
])

# Prevent transformers from enabling optional torchvision paths even if a runtime
# image later injects torchvision into site-packages.
os.environ["USE_TORCHVISION"] = "0"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

print("\nChecking torchvision visibility in this kernel env...")
print("torchvision spec:", importlib.util.find_spec("torchvision"))
print("\nIf torchvision spec is not None, restart may still fail; remove it manually from the active env.")
print("Dependencies ready. Restart the Jupyter kernel, then continue from STEP 1.")


Jupyter Python: /usr/local/bin/python
uv 0.5.14


Using Python 3.12.6 environment at: /usr/local
Resolved 77 packages in 473ms
Prepared 69 packages in 30.65s
Uninstalled 41 packages in 1.94s



Checking torchvision visibility in this kernel env...
torchvision spec: None

If torchvision spec is not None, restart may still fail; remove it manually from the active env.
Dependencies ready. Restart the Jupyter kernel, then continue from STEP 1.


Installed 69 packages in 552ms
 - accelerate==1.10.1
 + accelerate==1.13.0
 - aiohappyeyeballs==2.4.3
 + aiohappyeyeballs==2.6.2
 - aiohttp==3.10.8
 + aiohttp==3.14.1
 - aiosignal==1.3.1
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 - anyio==4.10.0
 + anyio==4.13.0
 - attrs==24.2.0
 + attrs==26.1.0
 - certifi==2024.8.30
 + certifi==2026.5.20
 - charset-normalizer==3.4.3
 + charset-normalizer==3.4.7
 - click==8.2.1
 + click==8.4.1
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.5.5
 + cuda-toolkit==13.0.2
 + datasets==5.0.0
 + dill==0.4.1
 - filelock==3.13.1
 + filelock==3.29.2
 - frozenlist==1.4.1
 + frozenlist==1.8.0
 - fsspec==2024.6.1
 + fsspec==2026.4.0
 - hf-xet==1.1.9
 + hf-xet==1.5.1
 - huggingface-hub==0.34.4
 + huggingface-hub==1.18.0
 - idna==3.10
 + idna==3.18
 - jinja2==3.1.4
 + jinja2==3.1.6
 - markdown-it-py==4.0.0
 + markdown-it-py==4.2.0
 - markupsafe==2.1.5
 + markupsafe==3.0.3
 - multidict==6.1.0
 + multidict==6.7.1
 + multiprocess==0.70.19
 - networkx==3.3
 + networkx=

In [1]:
# ========================================================
# STEP 1: Imports, environment checks, and TRL/vLLM compatibility guard
# ========================================================
# If you previously hit the torchvision::nms import error, make sure you ran
# the uv-based STEP 0 and restarted the kernel so partial imports are cleared.
# Keep torchvision disabled for this text-only training notebook.
import os
os.environ.setdefault("USE_TORCHVISION", "0")
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

import re
import sys
import gc
import ast
import json
import math
import time
import shutil
import random
import hashlib
import inspect
import tempfile
import subprocess
from pathlib import Path
from collections import Counter
from typing import Any, Optional, Tuple

import importlib.util
_tv_spec = importlib.util.find_spec("torchvision")
if _tv_spec is not None:
    raise RuntimeError(
        "torchvision is still installed in this kernel Python environment and may break transformers imports. "
        "Run STEP 0, confirm it uses the Python path printed there, then restart the kernel. "
        f"Detected at: {_tv_spec.origin}"
    )

import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, snapshot_download

# Some recent torch/transformers/trl combos expect this symbol.
if not hasattr(torch, "float8_e8m0fnu"):
    setattr(torch, "float8_e8m0fnu", torch.float32)

# Disable vLLM integration by default; many notebook runtimes have incompatible vLLM builds.
try:
    import importlib
    trl_import_utils = importlib.import_module("trl.import_utils")
    trl_import_utils.is_vllm_available = lambda: False
    trl_import_utils._vllm_available = False
except Exception:
    pass
sys.modules["vllm"] = None


def detect_attn_implementation() -> str:
    """Pick a safe attention implementation for the current accelerator."""
    if not torch.cuda.is_available():
        return "eager"
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else ""
    low = gpu_name.lower()
    is_amd = any(x in low for x in ["amd", "instinct", "radeon", "mi300", "mi250", "mi210"])
    if is_amd:
        try:
            from flash_attn import flash_attn_func  # noqa: F401
            return "flash_attention_2"
        except Exception:
            return "sdpa"
    # A100 / H100 / NVIDIA CUDA path.
    try:
        from flash_attn import flash_attn_func  # noqa: F401
        return "flash_attention_2"
    except Exception:
        print("flash-attn is not importable; falling back to sdpa. Install flash-attn on A100 for best throughput.")
        return "sdpa"

ATTN_IMPL = detect_attn_implementation()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Attention implementation: {ATTN_IMPL}")


flash-attn is not importable; falling back to sdpa. Install flash-attn on A100 for best throughput.
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
Attention implementation: sdpa


In [2]:
# ========================================================
# STEP 2: Configure Type-2 GRPO arguments
# ========================================================
import argparse


def str2bool(x):
    if isinstance(x, bool):
        return x
    return str(x).lower() in {"1", "true", "yes", "y", "on"}


def parse_args():
    parser = argparse.ArgumentParser(description="Type-2 Physics GRPO RL for Option-B Qwen single-model pipeline.")

    # Warm-start model. Switch this to the Qwen3 merged checkpoint when ready.
    parser.add_argument("--model_id", type=str, default="not-a-real-ai-guy/qwen2.5-type2-option-b-modes-merged")
    parser.add_argument("--alt_model_id_note", type=str, default="not-a-real-ai-guy/qwen3-type2-option-b-modes-merged")

    # Keep these _rl repos. Do not use the older SFT repos here.
    parser.add_argument("--gen_dataset_id", type=str, default="not-a-real-ai-guy/exact_2026_type2_generator_rl")
    parser.add_argument("--exp_dataset_id", type=str, default="not-a-real-ai-guy/exact_2026_type2_explainer_rl")

    # New RL adapter/output path, separate from SFT artifacts.
    parser.add_argument("--output_dir", type=str, default="models/qwen2.5-type2-option-b-modes-grpo-lora")
    parser.add_argument("--hub_model_id", type=str, default="not-a-real-ai-guy/qwen2.5-type2-option-b-modes-grpo-lora")

    # Optional: use a separate SFT LoRA adapter instead of a merged warm-start model.
    parser.add_argument("--sft_adapter_dir", type=str, default="", help="Optional SFT LoRA adapter. Leave blank when model_id is already merged.")

    # Data split / smoke-test knobs.
    parser.add_argument("--test_split", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--smoke_n", type=int, default=5)

    # GRPO/A100-friendly training knobs.
    parser.add_argument("--lr", type=float, default=1e-6)
    parser.add_argument("--beta", type=float, default=0.02)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--max_steps", type=int, default=-1, help="Set to 2 for a tiny GRPO smoke run; -1 for epoch-based training.")
    parser.add_argument("--per_device_train_batch_size", type=int, default=8)
    parser.add_argument("--grad_accum", type=int, default=1)
    parser.add_argument("--num_generations", type=int, default=4, help="Main speed/quality knob. 2 is fast; 4 is stronger but much slower.")
    parser.add_argument("--max_prompt_length", type=int, default=1024)
    parser.add_argument("--max_completion_length", type=int, default=512)
    parser.add_argument("--save_steps", type=int, default=100)
    parser.add_argument("--logging_steps", type=int, default=5)
    parser.add_argument("--sandbox_timeout_s", type=float, default=1.5)
    parser.add_argument("--sandbox_sample_rate", type=float, default=1.0, help="Fraction of completions that run the expensive subprocess sandbox reward. Use 1.0 for full-quality reward.")
    parser.add_argument("--enable_gradient_checkpointing", type=str2bool, default=False, help="False is faster on A100-80GB; turn on only if you hit VRAM OOM.")
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--hub_private_repo", type=str2bool, default=True)
    parser.add_argument("--push_to_hub", type=str2bool, default=False, help="Keep False during training for speed; push manually after a good checkpoint.")

    is_ipython = False
    try:
        from IPython import get_ipython
        is_ipython = get_ipython() is not None
    except Exception:
        pass
    if is_ipython or any("ipykernel" in arg for arg in sys.argv):
        return parser.parse_args(args=[])
    return parser.parse_args()

args = parse_args()
set_seed(args.seed)

# Hard guard: this notebook is for RL repos only.
assert args.gen_dataset_id.endswith("_rl"), f"Generator dataset must be an _rl repo, got {args.gen_dataset_id}"
assert args.exp_dataset_id.endswith("_rl"), f"Explainer dataset must be an _rl repo, got {args.exp_dataset_id}"

print("Type-2 GRPO configuration")
print(f"  model_id        : {args.model_id}")
print(f"  Qwen3 alternative: {args.alt_model_id_note}")
print(f"  gen_dataset_id  : {args.gen_dataset_id}")
print(f"  exp_dataset_id  : {args.exp_dataset_id}")
print(f"  output_dir      : {args.output_dir}")
print(f"  hub_model_id    : {args.hub_model_id}")
print(f"  max_steps       : {args.max_steps}  (-1 means epoch-based)")
print(f"  batch/gen/accum : {args.per_device_train_batch_size}/{args.num_generations}/{args.grad_accum}")
print(f"  max lengths     : prompt={args.max_prompt_length}, completion={args.max_completion_length}")
print(f"  sandbox sampling: {args.sandbox_sample_rate}")
print(f"  push_to_hub     : {args.push_to_hub}")


Type-2 GRPO configuration
  model_id        : not-a-real-ai-guy/qwen2.5-type2-option-b-modes-merged
  Qwen3 alternative: not-a-real-ai-guy/qwen3-type2-option-b-modes-merged
  gen_dataset_id  : not-a-real-ai-guy/exact_2026_type2_generator_rl
  exp_dataset_id  : not-a-real-ai-guy/exact_2026_type2_explainer_rl
  output_dir      : models/qwen2.5-type2-option-b-modes-grpo-lora
  hub_model_id    : not-a-real-ai-guy/qwen2.5-type2-option-b-modes-grpo-lora
  max_steps       : -1  (-1 means epoch-based)
  batch/gen/accum : 8/4/1
  max lengths     : prompt=1024, completion=512
  sandbox sampling: 1.0
  push_to_hub     : False


In [ ]:
# ========================================================
# STEP 3: Secure Hugging Face token input
# ========================================================
# Your token is read at runtime. It is not stored in the notebook.
import getpass

if hf_token:
    print("✓ HF token found in environment.")
else:
    print("No HF_TOKEN/HUGGINGFACE_HUB_TOKEN environment variable set.")
    hf_token = getpass.getpass("HF Token (hidden): ").strip()
    if not hf_token:
        raise RuntimeError("No Hugging Face token provided. Set HF_TOKEN or enter one here.")
    os.environ["HF_TOKEN"] = hf_token

login(token=hf_token)
print("✓ Logged in to Hugging Face Hub.")

✓ HF token found in environment.
✓ Logged in to Hugging Face Hub.


In [4]:
# ========================================================
# STEP 4: Type-2 Option-B prompt and dataset helper functions
# ========================================================
MODE_SYSTEM_PROMPTS = {
    "generate_code": (
        "You are a Type-2 physics solver in TASK_MODE=generate_code. "
        "Generate only safe executable Python/SymPy code for electric circuits, electrostatics, magnetism, and related physics problems. "
        "Return JSON only: {\"code\": \"<python code>\"}. "
        "The code may use only Python stdlib, json, math, fractions, decimal, and sympy. "
        "Define solve() returning a dict with keys answer, unit, kind, and derived. "
        "Use SI units internally. Print exactly one JSON object with print(json.dumps(solve(), ensure_ascii=False)). "
        "For computed Yes/No questions, compute the physics quantities first and set kind='boolean_computed'. "
        "For pure theory questions, still return code whose solve() returns kind='qualitative_theory'."
    ),
    "explain_from_result": (
        "You are a Type-2 physics solver in TASK_MODE=explain_from_result. "
        "Given the original question and a sandbox execution result, produce the final answer and a concise checkable explanation. "
        "Do not generate code. Return JSON only with keys answer, explanation, cot, premises, confidence."
    ),
}

CODE_JSON_SCHEMA = '{"code": "<python code>"}'


def _ensure_dict(x: Any) -> dict:
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            y = json.loads(x)
            return y if isinstance(y, dict) else {}
        except Exception:
            return {}
    return {}


def _content_to_str(x: Any) -> str:
    if isinstance(x, str):
        return x
    return json.dumps(x, ensure_ascii=False)


def _last_assistant_content(example: dict) -> str:
    msgs = example.get("messages", []) or []
    for m in reversed(msgs):
        if isinstance(m, dict) and m.get("role") == "assistant":
            return _content_to_str(m.get("content", "")).strip()
    # Fallback for datasets that store target directly.
    for key in ["assistant", "completion", "target", "response", "output"]:
        if example.get(key):
            return _content_to_str(example[key]).strip()
    return ""


def _extract_id(example: dict) -> str:
    for key in ["id", "problem_id", "sample_id"]:
        if example.get(key):
            return str(example[key])
    meta = _ensure_dict(example.get("metadata", {}))
    for key in ["id", "problem_id", "sample_id"]:
        if meta.get(key):
            return str(meta[key])
    return ""


def _extract_question(example: dict, meta: dict) -> str:
    for key in ["question", "problem", "prompt", "problem_statement"]:
        if meta.get(key):
            return str(meta[key]).strip()
        if example.get(key):
            return str(example[key]).strip()

    msgs = example.get("messages", []) or []
    for m in msgs:
        if not isinstance(m, dict) or m.get("role") != "user":
            continue
        content = _content_to_str(m.get("content", ""))
        for line in content.splitlines():
            if line.strip().lower().startswith("question:"):
                return line.split(":", 1)[1].strip()
        if len(content.strip()) > 20:
            return content.strip()
    return ""


def extract_json_object(text: str) -> dict:
    """Best-effort extraction of the first JSON object from model/dataset text."""
    if isinstance(text, dict):
        return text
    text = str(text or "").strip()
    if not text:
        raise ValueError("empty text")
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    fenced = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    for block in fenced:
        try:
            obj = json.loads(block)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue

    starts = [i for i, ch in enumerate(text) if ch == "{"]
    for start in starts:
        depth = 0
        in_str = False
        escape = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_str = False
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        candidate = text[start:i + 1]
                        try:
                            obj = json.loads(candidate)
                            if isinstance(obj, dict):
                                return obj
                        except Exception:
                            break
    raise ValueError("No valid JSON object found")


def build_generate_code_messages(pid: str, question: str) -> list[dict]:
    return [
        {"role": "system", "content": MODE_SYSTEM_PROMPTS["generate_code"]},
        {"role": "user", "content": (
            "TASK_MODE=generate_code\n"
            f"Problem ID: {pid}\n"
            f"Question: {question}\n\n"
            f"Return JSON only: {CODE_JSON_SCHEMA}"
        )},
    ]


def build_explain_from_result_messages(pid: str, question: str, runner_result: dict) -> list[dict]:
    answer_schema = '{"answer": "<final answer>", "explanation": "<concise explanation>", "cot": ["Step 1: ..."], "premises": ["formula/given"], "confidence": 0.0}'
    return [
        {"role": "system", "content": MODE_SYSTEM_PROMPTS["explain_from_result"]},
        {"role": "user", "content": (
            "TASK_MODE=explain_from_result\n"
            f"Problem ID: {pid}\n"
            f"Question: {question}\n\n"
            "Code execution result JSON:\n"
            f"{json.dumps(runner_result, ensure_ascii=False)}\n\n"
            f"Return JSON only: {answer_schema}"
        )},
    ]


def _runner_result_from_meta(gen_meta: dict, gen_example: dict) -> dict:
    for key in ["runner_result", "sandbox_result", "result", "execution_result"]:
        val = gen_meta.get(key, None)
        if val is not None:
            return _ensure_dict(val)
        val = gen_example.get(key, None)
        if val is not None:
            return _ensure_dict(val)
    return {}


def _flatten_runner_result(x: Any) -> dict:
    x = _ensure_dict(x)
    if x.get("status") == "ok" and isinstance(x.get("result"), dict):
        return x["result"]
    if isinstance(x.get("result"), dict) and ("answer" in x["result"] or "unit" in x["result"]):
        return x["result"]
    return x


def _extract_ref_kind(gen_meta: dict, exp_meta: dict, runner_result: dict) -> str:
    flat = _flatten_runner_result(runner_result)
    for obj in [flat, gen_meta, exp_meta]:
        for key in ["kind", "task_kind", "question_kind", "type"]:
            if isinstance(obj, dict) and obj.get(key):
                return str(obj[key])
    return ""

print("Prompt and dataset helpers ready.")


Prompt and dataset helpers ready.


In [5]:
# ========================================================
# STEP 5: Load tokenizer and merge the two Type-2 RL datasets by problem ID
# ========================================================
print(f"Loading tokenizer for {args.model_id}...")
tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Important for GRPO batched generation.

print(f"Loading generator RL dataset: {args.gen_dataset_id}")
gen_ds = load_dataset(args.gen_dataset_id, token=os.environ.get("HF_TOKEN"))
gen_split = gen_ds["train"] if "train" in gen_ds else list(gen_ds.values())[0]

print(f"Loading explainer RL dataset: {args.exp_dataset_id}")
exp_ds = load_dataset(args.exp_dataset_id, token=os.environ.get("HF_TOKEN"))
exp_split = exp_ds["train"] if "train" in exp_ds else list(exp_ds.values())[0]

# Materialize once; reward columns are easier to build as Python dicts.
gen_examples = [dict(x) for x in gen_split]
exp_examples = [dict(x) for x in exp_split]

gen_by_id = {_extract_id(x): x for x in gen_examples if _extract_id(x)}
exp_by_id = {_extract_id(x): x for x in exp_examples if _extract_id(x)}
matched_ids = sorted(set(gen_by_id) & set(exp_by_id))

print("\nMerge statistics")
print(f"  Generator records : {len(gen_examples)}")
print(f"  Explainer records : {len(exp_examples)}")
print(f"  Matched SYN IDs   : {len(matched_ids)}")
print(f"  Gen-only IDs      : {len(gen_by_id) - len(matched_ids)}")
print(f"  Exp-only IDs      : {len(exp_by_id) - len(matched_ids)}")

problem_records = []
skip_reasons = Counter()

for pid in matched_ids:
    gen_example = gen_by_id[pid]
    exp_example = exp_by_id[pid]
    gen_meta = _ensure_dict(gen_example.get("metadata", {}))
    exp_meta = _ensure_dict(exp_example.get("metadata", {}))

    question = _extract_question(gen_example, gen_meta) or _extract_question(exp_example, exp_meta)
    code_target = _last_assistant_content(gen_example)
    explainer_target = _last_assistant_content(exp_example)
    runner_result = _runner_result_from_meta(gen_meta, gen_example)
    flat_runner = _flatten_runner_result(runner_result)

    if not question:
        skip_reasons["missing_question"] += 1
        continue
    if not code_target:
        skip_reasons["missing_generator_target"] += 1
        continue
    if not explainer_target:
        skip_reasons["missing_explainer_target"] += 1
        continue

    try:
        code_target_json = extract_json_object(code_target)
    except Exception:
        code_target_json = {"raw": code_target}

    try:
        explainer_json = extract_json_object(explainer_target)
    except Exception:
        explainer_json = {"raw": explainer_target}

    gold_answer = (
        gen_meta.get("gold_answer") or exp_meta.get("gold_answer") or
        gen_meta.get("answer") or exp_meta.get("answer") or
        flat_runner.get("answer") or explainer_json.get("answer") or ""
    )
    gold_unit = (
        gen_meta.get("gold_unit") or exp_meta.get("gold_unit") or
        gen_meta.get("unit") or exp_meta.get("unit") or
        flat_runner.get("unit") or ""
    )
    ref_kind = _extract_ref_kind(gen_meta, exp_meta, runner_result)

    messages = build_generate_code_messages(pid, question)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    problem_records.append({
        "problem_id": pid,
        "question": question,
        "prompt": prompt_text,
        "messages_json": json.dumps(messages, ensure_ascii=False),
        "ref_code_json": json.dumps(code_target_json, ensure_ascii=False),
        "ref_code_text": code_target,
        "ref_runner_result_json": json.dumps(runner_result, ensure_ascii=False),
        "ref_flat_runner_json": json.dumps(flat_runner, ensure_ascii=False),
        "ref_explainer_json": json.dumps(explainer_json, ensure_ascii=False),
        "ref_explainer_text": explainer_target,
        "ref_gold_answer": str(gold_answer),
        "ref_gold_unit": str(gold_unit),
        "ref_kind": ref_kind,
    })

print("\nProblem-level filtering")
print(f"  Kept problem pairs : {len(problem_records)}")
print(f"  Skipped            : {sum(skip_reasons.values())}")
for k, v in skip_reasons.most_common():
    print(f"    - {k}: {v}")

rng = random.Random(args.seed)
rng.shuffle(problem_records)
if len(problem_records) >= 10 and args.test_split > 0:
    n_test = max(1, int(round(len(problem_records) * args.test_split)))
else:
    n_test = 0

eval_records = problem_records[:n_test]
train_records = problem_records[n_test:]

train_ids = {x["problem_id"] for x in train_records}
eval_ids = {x["problem_id"] for x in eval_records}
assert train_ids.isdisjoint(eval_ids), "Leakage: same problem ID appears in train and eval."

rl_dataset = DatasetDict({"train": Dataset.from_list(train_records)})
if eval_records:
    rl_dataset["test"] = Dataset.from_list(eval_records)

train_dataset = rl_dataset["train"]
eval_dataset = rl_dataset.get("test", None)

print("\nRL dataset ready")
print(f"  Train problems: {len(train_dataset)}")
print(f"  Eval problems : {len(eval_dataset) if eval_dataset is not None else 0}")
print("  Columns       :", train_dataset.column_names)

print("\nSample prompt")
print(train_dataset[0]["prompt"][:1200])
print("\nReference code target preview")
print(train_dataset[0]["ref_code_text"][:800])


Loading tokenizer for not-a-real-ai-guy/qwen2.5-type2-option-b-modes-merged...


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading explainer RL dataset: not-a-real-ai-guy/exact_2026_type2_explainer_rl


README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

explainer.jsonl:   0%|          | 0.00/5.04M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]


Merge statistics
  Generator records : 946
  Explainer records : 946
  Matched SYN IDs   : 946
  Gen-only IDs      : 0
  Exp-only IDs      : 0

Problem-level filtering
  Kept problem pairs : 946
  Skipped            : 0

RL dataset ready
  Train problems: 899
  Eval problems : 47
  Columns       : ['problem_id', 'question', 'prompt', 'messages_json', 'ref_code_json', 'ref_code_text', 'ref_runner_result_json', 'ref_flat_runner_json', 'ref_explainer_json', 'ref_explainer_text', 'ref_gold_answer', 'ref_gold_unit', 'ref_kind']

Sample prompt
<|im_start|>system
You are a Type-2 physics solver in TASK_MODE=generate_code. Generate only safe executable Python/SymPy code for electric circuits, electrostatics, magnetism, and related physics problems. Return JSON only: {"code": "<python code>"}. The code may use only Python stdlib, json, math, fractions, decimal, and sympy. Define solve() returning a dict with keys answer, unit, kind, and derived. Use SI units internally. Print exactly one JSON 

In [6]:
# ========================================================
# STEP 6: Safe Python/SymPy sandbox used by RL rewards
# ========================================================
ALLOWED_IMPORTS = {"json", "math", "sympy", "fractions", "decimal"}
BANNED_CALLS = {
    "open", "exec", "eval", "compile", "__import__", "input", "breakpoint",
    "getattr", "setattr", "delattr", "globals", "locals", "vars"
}
BANNED_ATTRS = {"system", "popen", "spawn", "fork", "remove", "unlink", "rmdir", "mkdir", "makedirs"}


def validate_code_ast(code: str) -> None:
    """Reject unsafe generated code before subprocess execution."""
    if not code or not str(code).strip():
        raise ValueError("empty code")
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = []
            if isinstance(node, ast.Import):
                names = [alias.name.split(".")[0] for alias in node.names]
            else:
                names = [(node.module or "").split(".")[0]]
            for name in names:
                if name not in ALLOWED_IMPORTS:
                    raise ValueError(f"Disallowed import: {name}")
        if isinstance(node, ast.Call):
            func = node.func
            name = None
            if isinstance(func, ast.Name):
                name = func.id
            elif isinstance(func, ast.Attribute):
                name = func.attr
            if name in BANNED_CALLS or name in BANNED_ATTRS:
                raise ValueError(f"Disallowed call: {name}")


def _limit_subprocess_resources():
    """Best-effort Linux resource limits for sandbox subprocesses."""
    try:
        import resource
        # 512 MB address space, 5s CPU. Ignore failures on unsupported systems.
        resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
        resource.setrlimit(resource.RLIMIT_CPU, (5, 5))
    except Exception:
        pass


def run_sandbox(code: str, timeout_s: Optional[float] = None) -> dict:
    """
    Execute generated Python/SymPy code in a subprocess.
    Returns {"status": "ok", "result": dict} or {"status": "error", ...}.
    """
    timeout_s = float(timeout_s or args.sandbox_timeout_s)
    try:
        validate_code_ast(code)
    except Exception as e:
        return {"status": "error", "error": f"AST validation failed: {e}"}

    wrapper = f"""
import json, math
import sympy as sp
sympy = sp

{code}

# Force exactly one final JSON line from solve() when possible.
if "solve" in globals() and callable(globals()["solve"]):
    _out = solve()
    print(json.dumps(_out, ensure_ascii=False))
"""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(wrapper)
        tmp_path = f.name

    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout_s,
            preexec_fn=_limit_subprocess_resources if os.name == "posix" else None,
        )
        if proc.returncode != 0:
            return {
                "status": "error",
                "error": "Sandbox process failed",
                "stderr": proc.stderr[-2000:],
                "stdout": proc.stdout[-2000:],
            }
        lines = [ln.strip() for ln in proc.stdout.splitlines() if ln.strip()]
        if not lines:
            return {"status": "error", "error": "No output produced by code"}
        for line in reversed(lines):
            try:
                parsed = json.loads(line)
                if isinstance(parsed, dict):
                    return {"status": "ok", "result": parsed, "stdout_tail": proc.stdout[-2000:]}
            except Exception:
                continue
        return {"status": "error", "error": "No JSON object found in stdout", "stdout": proc.stdout[-2000:]}
    except subprocess.TimeoutExpired:
        return {"status": "error", "error": f"Sandbox timeout after {timeout_s}s"}
    except Exception as e:
        return {"status": "error", "error": f"Sandbox error: {e}"}
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass

print("Safe sandbox ready.")


Safe sandbox ready.


In [8]:
# ========================================================
# STEP 7: Unit-aware equivalence and reward helpers
# ========================================================
_UNIT_TABLE = {
    "": ("dimensionless", 1.0), "unitless": ("dimensionless", 1.0), "—": ("dimensionless", 1.0), "-": ("dimensionless", 1.0),
    # voltage/current/resistance/power/energy
    "v": ("voltage", 1.0), "mv": ("voltage", 1e-3), "kv": ("voltage", 1e3),
    "a": ("current", 1.0), "ma": ("current", 1e-3), "ua": ("current", 1e-6), "μa": ("current", 1e-6),
    "ohm": ("resistance", 1.0), "ω": ("resistance", 1.0), "Ω": ("resistance", 1.0), "kohm": ("resistance", 1e3), "kω": ("resistance", 1e3), "kΩ": ("resistance", 1e3),
    "w": ("power", 1.0), "mw": ("power", 1e-3), "kw": ("power", 1e3),
    "j": ("energy", 1.0), "mj": ("energy", 1e-3), "kj": ("energy", 1e3),
    # capacitance/inductance/charge/frequency
    "f": ("capacitance", 1.0), "mf": ("capacitance", 1e-3), "uf": ("capacitance", 1e-6), "μf": ("capacitance", 1e-6), "nf": ("capacitance", 1e-9), "pf": ("capacitance", 1e-12),
    "h": ("inductance", 1.0), "mh": ("inductance", 1e-3), "uh": ("inductance", 1e-6), "μh": ("inductance", 1e-6),
    "c": ("charge", 1.0), "mc": ("charge", 1e-3), "uc": ("charge", 1e-6), "μc": ("charge", 1e-6), "nc": ("charge", 1e-9), "pc": ("charge", 1e-12),
    "hz": ("frequency", 1.0), "khz": ("frequency", 1e3), "mhz": ("frequency", 1e6),
    # field/force/magnetism
    "n": ("force", 1.0), "mn": ("force", 1e-3),
    "v/m": ("electric_field", 1.0), "n/c": ("electric_field", 1.0), "v m^-1": ("electric_field", 1.0),
    "t": ("magnetic_field", 1.0), "mt": ("magnetic_field", 1e-3), "ut": ("magnetic_field", 1e-6), "μt": ("magnetic_field", 1e-6),
    "wb": ("magnetic_flux", 1.0), "mwb": ("magnetic_flux", 1e-3),
    # angle
    "rad": ("angle", 1.0), "deg": ("angle", math.pi / 180), "degree": ("angle", math.pi / 180), "degrees": ("angle", math.pi / 180),
}

_UNIT_ALIASES = {
    "volt": "v", "volts": "v", "amp": "a", "amps": "a", "ampere": "a", "amperes": "a",
    "ohms": "ohm", "omega": "ohm", "watt": "w", "watts": "w", "joule": "j", "joules": "j",
    "farad": "f", "farads": "f", "henry": "h", "henrys": "h", "coulomb": "c", "coulombs": "c",
    "newton": "n", "newtons": "n", "tesla": "t", "teslas": "t", "weber": "wb", "webers": "wb",
}


def normalize_unit_text(unit: Any) -> str:
    if unit is None:
        return ""
    u = str(unit).strip().replace("Ω", "Ω").replace("µ", "μ")
    u = u.replace("Ω", "ω").replace("μ", "u").replace(" ", "").replace("−", "-").lower()
    u = u.replace("per", "/")
    return _UNIT_ALIASES.get(u, u)


def unit_info(unit: Any) -> Optional[Tuple[str, float]]:
    u = normalize_unit_text(unit)
    return _UNIT_TABLE.get(u)


def parse_first_number(x: Any) -> Optional[float]:
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    s = s.replace(",", "").replace("−", "-").replace("×", "x").replace("·", "*")
    s = re.sub(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*(?:x|\*)\s*10\s*\^?\s*\(?\s*([+-]?\d+)\s*\)?",
        r"\1e\2",
        s,
        flags=re.I,
    )
    m = re.search(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", s)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None


def maybe_extract_unit_from_answer(answer: Any) -> str:
    s = str(answer or "").strip()
    m = re.search(r"([A-Za-zμµΩΩ/^\-]+)\s*$", s)
    return m.group(1) if m else ""


def normalize_text_answer(x: Any) -> str:
    s = str(x or "").strip().lower()
    s = s.replace("−", "-")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9+\-./ ]+", "", s)
    return s.strip()


def rule_based_equivalence(pred_answer, pred_unit, ref_answer, ref_unit, rtol=1e-3, atol=1e-8) -> dict:
    pred_num = parse_first_number(pred_answer)
    ref_num = parse_first_number(ref_answer)
    pred_unit = pred_unit or maybe_extract_unit_from_answer(pred_answer)
    ref_unit = ref_unit or maybe_extract_unit_from_answer(ref_answer)
    pinfo = unit_info(pred_unit)
    rinfo = unit_info(ref_unit)

    out = {"equivalent": None, "score": 0.0, "category": "unknown", "reason": ""}

    if pred_num is not None and ref_num is not None and pinfo and rinfo:
        pdim, pf = pinfo
        rdim, rf = rinfo
        if pdim != rdim:
            out.update(equivalent=False, category="wrong_unit_dimension", reason=f"{pdim} != {rdim}")
            return out
        p_si = pred_num * pf
        r_si = ref_num * rf
        ok = abs(p_si - r_si) <= max(atol, rtol * max(abs(r_si), 1e-12))
        out.update(equivalent=bool(ok), score=1.0 if ok else 0.0, category="numeric_unit_equivalent" if ok else "wrong_numeric_value", reason=f"normalized {pdim}")
        return out

    if pred_num is not None and ref_num is not None and not str(ref_unit).strip() and not str(pred_unit).strip():
        ok = abs(pred_num - ref_num) <= max(atol, rtol * max(abs(ref_num), 1e-12))
        out.update(equivalent=bool(ok), score=1.0 if ok else 0.0, category="numeric_equivalent_no_unit" if ok else "wrong_numeric_value_no_unit")
        return out

    pt = normalize_text_answer(pred_answer)
    rt = normalize_text_answer(ref_answer)
    if pt and rt:
        if pt == rt or pt in rt or rt in pt:
            out.update(equivalent=True, score=1.0, category="text_equivalent")
            return out
        # Yes/no aliases.
        yes = {"yes", "true", "occurs", "resonance occurs"}
        no = {"no", "false", "does not occur", "not occur", "no resonance"}
        if (pt in yes and rt in yes) or (pt in no and rt in no):
            out.update(equivalent=True, score=1.0, category="boolean_text_equivalent")
            return out

    out["reason"] = "No confident rule-based equivalence."
    return out


def json_loads_maybe(x: Any) -> dict:
    if isinstance(x, dict):
        return x
    return _ensure_dict(x)

print("Unit-aware equivalence helpers ready.")


Unit-aware equivalence helpers ready.


In [9]:
# ========================================================
# STEP 8: Physics-specific GRPO reward functions
# ========================================================
_SANDBOX_CACHE = {}


def completion_to_text(completion: Any) -> str:
    """TRL may pass completion strings or chat-message lists depending on version/config."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict):
                parts.append(str(item.get("content", "")))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    if isinstance(completion, dict):
        return str(completion.get("content", completion))
    return str(completion)


def parse_code_completion(completion: Any) -> dict:
    text = completion_to_text(completion)
    try:
        obj = extract_json_object(text)
        code = obj.get("code", "") if isinstance(obj, dict) else ""
        return {"ok": bool(code), "json": obj, "code": str(code), "text": text, "error": "" if code else "missing code"}
    except Exception as e:
        return {"ok": False, "json": {}, "code": "", "text": text, "error": str(e)}


def _stable_fraction(*parts: Any) -> float:
    raw = "||".join(str(p) for p in parts)
    h = hashlib.sha256(raw.encode("utf-8", errors="ignore")).hexdigest()[:12]
    return int(h, 16) / float(16 ** 12)


def should_run_sandbox(i: int, completion: Any, kwargs: dict) -> bool:
    """Deterministically sample expensive sandbox rewards.

    With sandbox_sample_rate < 1, only a subset of completions run subprocess+SymPy.
    Non-sampled completions still receive JSON/compactness rewards. This trades reward
    density for much faster training on CPU-starved A100 notebook profiles.
    """
    rate = max(0.0, min(1.0, float(getattr(args, "sandbox_sample_rate", 1.0))))
    if rate >= 1.0:
        return True
    pid = _col(kwargs, "problem_id", i, "")
    return _stable_fraction(args.seed, pid, completion_to_text(completion)) < rate


def eval_completion_cached(completion: Any) -> dict:
    text = completion_to_text(completion)
    key = hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()
    if key in _SANDBOX_CACHE:
        return _SANDBOX_CACHE[key]
    parsed = parse_code_completion(text)
    sandbox = run_sandbox(parsed["code"]) if parsed["ok"] else {"status": "error", "error": parsed["error"]}
    out = {"parsed": parsed, "sandbox": sandbox}
    _SANDBOX_CACHE[key] = out
    return out


def _col(kwargs: dict, name: str, i: int, default=None):
    vals = kwargs.get(name, None)
    if vals is None:
        return default
    try:
        return vals[i]
    except Exception:
        return default


def _reference_answer_unit(i: int, kwargs: dict) -> tuple[str, str, dict, dict]:
    flat_runner = json_loads_maybe(_col(kwargs, "ref_flat_runner_json", i, "{}"))
    runner = json_loads_maybe(_col(kwargs, "ref_runner_result_json", i, "{}"))
    explainer = json_loads_maybe(_col(kwargs, "ref_explainer_json", i, "{}"))
    gold_answer = _col(kwargs, "ref_gold_answer", i, "") or ""
    gold_unit = _col(kwargs, "ref_gold_unit", i, "") or ""

    ref_answer = flat_runner.get("answer") or gold_answer or explainer.get("answer") or ""
    ref_unit = flat_runner.get("unit") or gold_unit or ""
    return str(ref_answer), str(ref_unit), runner, explainer


def json_code_format_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        parsed = parse_code_completion(completion)
        if parsed["ok"]:
            # Small extra if the output is only JSON-ish, not prose + JSON.
            stripped = parsed["text"].strip()
            rewards.append(1.0 if stripped.startswith("{") and stripped.endswith("}") else 0.6)
        else:
            rewards.append(-1.0)
    return rewards


def sandbox_execution_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        if not should_run_sandbox(i, completion, kwargs):
            rewards.append(0.0)
            continue
        ev = eval_completion_cached(completion)
        rewards.append(2.0 if ev["sandbox"].get("status") == "ok" else -2.0)
    return rewards


def answer_correctness_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        if not should_run_sandbox(i, completion, kwargs):
            rewards.append(0.0)
            continue
        ev = eval_completion_cached(completion)
        sandbox = ev["sandbox"]
        if sandbox.get("status") != "ok":
            rewards.append(-2.5)
            continue

        pred = sandbox.get("result", {}) if isinstance(sandbox.get("result"), dict) else {}
        pred_answer = pred.get("answer", "")
        pred_unit = pred.get("unit", "")
        ref_answer, ref_unit, _, explainer = _reference_answer_unit(i, kwargs)

        # First compare to stored runner/gold. If inconclusive, compare to explainer answer.
        eq = rule_based_equivalence(pred_answer, pred_unit, ref_answer, ref_unit)
        if eq.get("equivalent") is True:
            rewards.append(4.0)
        elif eq.get("equivalent") is False:
            rewards.append(-2.5)
        else:
            exp_ans = explainer.get("answer", "") if isinstance(explainer, dict) else ""
            exp_eq = rule_based_equivalence(pred_answer, pred_unit, exp_ans, ref_unit)
            if exp_eq.get("equivalent") is True:
                rewards.append(3.0)
            elif pred_answer not in [None, ""]:
                rewards.append(0.5)
            else:
                rewards.append(-1.5)
    return rewards


def kind_structure_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        if not should_run_sandbox(i, completion, kwargs):
            rewards.append(0.0)
            continue
        ev = eval_completion_cached(completion)
        sandbox = ev["sandbox"]
        if sandbox.get("status") != "ok":
            rewards.append(-0.5)
            continue
        result = sandbox.get("result", {}) if isinstance(sandbox.get("result"), dict) else {}
        score = 0.0
        for key in ["answer", "unit", "kind", "derived"]:
            if key in result:
                score += 0.25
        if isinstance(result.get("derived", {}), dict):
            score += 0.25
        rewards.append(min(score, 1.0))
    return rewards


def physics_consistency_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        if not should_run_sandbox(i, completion, kwargs):
            rewards.append(0.0)
            continue
        ev = eval_completion_cached(completion)
        sandbox = ev["sandbox"]
        if sandbox.get("status") != "ok":
            rewards.append(0.0)
            continue
        result = sandbox.get("result", {}) if isinstance(sandbox.get("result"), dict) else {}
        pred_kind = normalize_text_answer(result.get("kind", ""))
        ref_kind = normalize_text_answer(_col(kwargs, "ref_kind", i, ""))
        pred_unit = result.get("unit", "")
        _, ref_unit, _, _ = _reference_answer_unit(i, kwargs)

        score = 0.0
        if pred_kind and ref_kind:
            score += 0.5 if (pred_kind == ref_kind or pred_kind in ref_kind or ref_kind in pred_kind) else -0.25
        pinfo, rinfo = unit_info(pred_unit), unit_info(ref_unit)
        if pinfo and rinfo:
            score += 0.5 if pinfo[0] == rinfo[0] else -0.75
        rewards.append(score)
    return rewards


def compactness_reward_func(prompts, completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        parsed = parse_code_completion(completion)
        code = parsed.get("code", "")
        n_chars = len(code)
        if not code.strip():
            rewards.append(-0.5)
        elif n_chars < 4000 and "def solve" in code and "json.dumps" in code:
            rewards.append(0.5)
        elif n_chars > 10000:
            rewards.append(-1.0)
        else:
            rewards.append(0.0)
    return rewards

reward_funcs = [
    json_code_format_reward_func,
    sandbox_execution_reward_func,
    answer_correctness_reward_func,
    kind_structure_reward_func,
    physics_consistency_reward_func,
    compactness_reward_func,
]

print(f"Registered {len(reward_funcs)} Type-2 physics reward functions.")
print(f"Sandbox sample rate: {args.sandbox_sample_rate} (set to 1.0 for full reward quality)")


Registered 6 Type-2 physics reward functions.
Sandbox sample rate: 1.0 (set to 1.0 for full reward quality)


In [10]:
# ========================================================
# STEP 9: Smoke-test parsing, sandbox, and rewards on stored reference targets
# ========================================================
# This uses the dataset's own generator target as a completion. It does not train yet.
smoke_n = min(args.smoke_n, len(train_dataset))
smoke_rows = [train_dataset[i] for i in range(smoke_n)]
smoke_prompts = [r["prompt"] for r in smoke_rows]
smoke_completions = [r["ref_code_text"] for r in smoke_rows]
smoke_kwargs = {name: [r[name] for r in smoke_rows] for name in train_dataset.column_names if name not in ["prompt"]}

print(f"Running reward smoke test on {smoke_n} stored reference completions...")
for fn in reward_funcs:
    scores = fn(smoke_prompts, smoke_completions, **smoke_kwargs)
    print(f"  {fn.__name__:35s}: {scores} | mean={sum(scores)/max(len(scores),1):.3f}")

# Inspect one sandbox result.
if smoke_completions:
    ev = eval_completion_cached(smoke_completions[0])
    print("\nExample parsed code OK:", ev["parsed"]["ok"])
    print("Example sandbox status:", ev["sandbox"].get("status"))
    print("Example sandbox result:", json.dumps(ev["sandbox"].get("result", {}), ensure_ascii=False)[:1000])


Running reward smoke test on 5 stored reference completions...
  json_code_format_reward_func       : [1.0, 1.0, 1.0, 1.0, 1.0] | mean=1.000
  sandbox_execution_reward_func      : [2.0, 2.0, 2.0, 2.0, 2.0] | mean=2.000
  answer_correctness_reward_func     : [4.0, 4.0, 4.0, 4.0, 4.0] | mean=4.000
  kind_structure_reward_func         : [1.0, 1.0, 1.0, 1.0, 1.0] | mean=1.000
  physics_consistency_reward_func    : [1.0, 1.0, 1.0, 1.0, 1.0] | mean=1.000
  compactness_reward_func            : [0.5, 0.5, 0.5, 0.5, 0.5] | mean=0.500

Example parsed code OK: True
Example sandbox status: ok
Example sandbox result: {"answer": "Net electric field is zero on the side of the negative charge (Q2) outside the region between the charges, farther from the positive charge.", "unit": "—", "kind": "qualitative", "derived": "For two opposite charges with Q1 > Q2, the zero-field point lies beyond the smaller charge (Q2) because the fields from Q1 and Q2 are opposite in direction there, and the condition for 

In [11]:
# ========================================================
# STEP 10: Load warm-start model and attach trainable RL LoRA adapter
# ========================================================
print(f"Loading warm-start model: {args.model_id}")
model = AutoModelForCausalLM.from_pretrained(
    args.model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=ATTN_IMPL,
)
model.config.use_cache = False

if args.sft_adapter_dir:
    print(f"Applying existing SFT adapter as trainable base: {args.sft_adapter_dir}")
    model = PeftModel.from_pretrained(model, args.sft_adapter_dir, is_trainable=True)
else:
    print("Attaching a fresh RL LoRA adapter on top of the merged SFT model.")
    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)

if args.enable_gradient_checkpointing:
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    except Exception as e:
        print(f"Gradient checkpointing not enabled: {e}")
else:
    print("Gradient checkpointing disabled for speed. Enable only if you hit VRAM OOM.")

model.print_trainable_parameters()
print("Model ready for GRPO.")


Loading warm-start model: not-a-real-ai-guy/qwen2.5-type2-option-b-modes-merged


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Attaching a fresh RL LoRA adapter on top of the merged SFT model.
Gradient checkpointing disabled for speed. Enable only if you hit VRAM OOM.
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Model ready for GRPO.


In [13]:
# ========================================================
# STEP 11: Configure GRPOTrainer with version-compatible kwargs and resume logic
# ========================================================
class GRPOLogCallback(TrainerCallback):
    def on_log(self, args_, state, control, logs=None, **kwargs):
        if not logs:
            return
        print(f"\n=== GRPO step {state.global_step} ===")
        for key in ["loss", "learning_rate", "objective/kl", "completions/lengths"]:
            if key in logs:
                val = logs[key]
                print(f"  {key}: {val:.6g}" if isinstance(val, (int, float)) else f"  {key}: {val}")
        reward_keys = sorted(k for k in logs if k.startswith("rewards/"))
        if reward_keys:
            print("  rewards:")
            for k in reward_keys:
                print(f"    {k.replace('rewards/', '')}: {logs[k]:.4f}")
        print("==============================")


def filter_kwargs(cls_or_func, kwargs: dict, label: str) -> dict:
    try:
        if hasattr(cls_or_func, "__dataclass_fields__"):
            allowed = set(cls_or_func.__dataclass_fields__.keys())
        else:
            allowed = set(inspect.signature(cls_or_func).parameters.keys())
    except Exception:
        return kwargs
    dropped = sorted(k for k in kwargs if k not in allowed)
    if dropped:
        print(f"[WARN] Dropping unsupported {label} kwargs: {dropped}")
    return {k: v for k, v in kwargs.items() if k in allowed}

config_kwargs = {
    "output_dir": args.output_dir,
    "learning_rate": args.lr,
    "beta": args.beta,
    "num_train_epochs": args.epochs,
    "max_steps": args.max_steps,
    "per_device_train_batch_size": args.per_device_train_batch_size,
    "gradient_accumulation_steps": args.grad_accum,
    "num_generations": args.num_generations,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": args.logging_steps,
    "bf16": bool(torch.cuda.is_available()),
    "save_strategy": "steps",
    "save_steps": args.save_steps,
    "save_total_limit": 2,
    "push_to_hub": args.push_to_hub,
    "hub_model_id": args.hub_model_id,
    "hub_strategy": "checkpoint",
    "hub_token": os.environ.get("HF_TOKEN"),
    "hub_private_repo": args.hub_private_repo,
    "report_to": "none",
    "seed": args.seed,
    "remove_unused_columns": False,
}

# Length fields moved across TRL versions; keep them version-compatible.
for key, val in {
    "max_prompt_length": args.max_prompt_length,
    "max_completion_length": args.max_completion_length,
}.items():
    config_kwargs[key] = val

# Prefer no vLLM inside this notebook unless manually re-enabled in a known-good runtime.
config_kwargs["use_vllm"] = False

trainer_kwargs = {
    "model": model,
    "reward_funcs": reward_funcs,
    "train_dataset": train_dataset,
    "eval_dataset": eval_dataset,
    "processing_class": tokenizer,
    "callbacks": [GRPOLogCallback()],
}

# Some TRL versions use tokenizer= instead of processing_class=.
trainer_params = set(inspect.signature(GRPOTrainer.__init__).parameters.keys())
if "processing_class" not in trainer_params and "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = trainer_kwargs.pop("processing_class")

config_kwargs = filter_kwargs(GRPOConfig, config_kwargs, "GRPOConfig")
trainer_kwargs = filter_kwargs(GRPOTrainer.__init__, trainer_kwargs, "GRPOTrainer")

print("Initializing GRPOTrainer...")
training_args = GRPOConfig(**config_kwargs)
trainer = GRPOTrainer(args=training_args, **trainer_kwargs)

# Smart checkpoint resume: local first, then Hub.
last_ckpt = None
if os.path.isdir(args.output_dir):
    last_ckpt = get_last_checkpoint(args.output_dir)
    if last_ckpt:
        print(f"Found LOCAL checkpoint: {last_ckpt}")

if last_ckpt is None and args.push_to_hub:
    try:
        print(f"No local checkpoint. Trying Hub checkpoint pull: {args.hub_model_id}")
        snapshot_download(
            repo_id=args.hub_model_id,
            token=os.environ.get("HF_TOKEN"),
            local_dir=args.output_dir,
            local_dir_use_symlinks=False,
        )
        last_ckpt = get_last_checkpoint(args.output_dir)
        if last_ckpt:
            print(f"Found HUB checkpoint: {last_ckpt}")
    except Exception as e:
        print(f"Hub checkpoint pull skipped: {e}")

print(f"Resume checkpoint: {last_ckpt}")
print("Trainer ready.")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[WARN] Dropping unsupported GRPOConfig kwargs: ['max_prompt_length']
Initializing GRPOTrainer...


[RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Resume checkpoint: None
Trainer ready.


In [15]:
# ========================================================
# STEP 12: Run GRPO training
# ========================================================
# For a tiny smoke run, set args.max_steps = 2 in the config cell and rerun cells 11-12.
print("Starting Type-2 GRPO training...")
trainer.train(resume_from_checkpoint=last_ckpt)
print("Training complete.")


Starting Type-2 GRPO training...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ========================================================
# STEP 13: Post-GRPO validation on held-out test set
# ========================================================
# This evaluates the trained RL adapter on held-out SYN problem IDs only.
# It does the real pipeline:
#   prompt -> model generates {"code": ...} -> sandbox runs code -> answer/unit equivalence check
#
# Set POST_GRPO_EVAL_LIMIT = 0 for full held-out validation.
# Keep it small first to sanity-check runtime.
from tqdm.auto import tqdm

POST_GRPO_EVAL_LIMIT = 0          # 0 = full eval_dataset; e.g. 20 for quick validation
POST_GRPO_BATCH_SIZE = 2          # Increase to 4/8 if VRAM allows; CPU sandbox may still bottleneck
POST_GRPO_MAX_NEW_TOKENS = min(int(args.max_completion_length), 512)
POST_GRPO_RESULTS_PATH = Path(args.output_dir) / "validation_results_grpo.jsonl"

if eval_dataset is None or len(eval_dataset) == 0:
    raise RuntimeError("No held-out eval_dataset found. Increase --test_split in STEP 2 and rerun dataset loading.")

# Prefer the trainer model because it contains the trained RL LoRA adapter after trainer.train().
eval_model = getattr(trainer, "model", model)
eval_model.eval()

try:
    eval_device = next(eval_model.parameters()).device
except Exception:
    eval_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

n_eval = len(eval_dataset) if POST_GRPO_EVAL_LIMIT in [0, None] else min(int(POST_GRPO_EVAL_LIMIT), len(eval_dataset))
eval_rows = [dict(eval_dataset[i]) for i in range(n_eval)]
POST_GRPO_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)


def _row_reference_answer_unit(row: dict) -> tuple[str, str, dict, dict]:
    flat_runner = json_loads_maybe(row.get("ref_flat_runner_json", "{}"))
    runner = json_loads_maybe(row.get("ref_runner_result_json", "{}"))
    explainer = json_loads_maybe(row.get("ref_explainer_json", "{}"))
    gold_answer = row.get("ref_gold_answer", "") or ""
    gold_unit = row.get("ref_gold_unit", "") or ""

    ref_answer = flat_runner.get("answer") or gold_answer or explainer.get("answer") or ""
    ref_unit = flat_runner.get("unit") or gold_unit or ""
    return str(ref_answer), str(ref_unit), runner, explainer


def _generate_code_completions(rows: list[dict]) -> list[str]:
    prompts = [r["prompt"] for r in rows]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=int(args.max_prompt_length),
    )
    inputs = {k: v.to(eval_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = eval_model.generate(
            **inputs,
            max_new_tokens=int(POST_GRPO_MAX_NEW_TOKENS),
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Since the batch is padded to a fixed input length, generated tokens start after this index.
    prompt_width = inputs["input_ids"].shape[1]
    gen_only = output_ids[:, prompt_width:]
    return tokenizer.batch_decode(gen_only, skip_special_tokens=True)


results = []
metric = Counter()
unit_dim_num = 0
unit_dim_den = 0
kind_match_num = 0
kind_match_den = 0
code_chars_total = 0

print(f"Running post-GRPO validation on {n_eval} held-out examples...")
print(f"Results will be saved to: {POST_GRPO_RESULTS_PATH}")

with open(POST_GRPO_RESULTS_PATH, "w", encoding="utf-8") as f:
    for start in tqdm(range(0, n_eval, POST_GRPO_BATCH_SIZE)):
        batch = eval_rows[start:start + POST_GRPO_BATCH_SIZE]
        completions = _generate_code_completions(batch)

        for row, completion in zip(batch, completions):
            pid = row.get("problem_id", "")
            question = row.get("question", "")
            ref_answer, ref_unit, ref_runner, ref_explainer = _row_reference_answer_unit(row)

            parsed = parse_code_completion(completion)
            code = parsed.get("code", "") or ""
            code_chars_total += len(code)

            if parsed.get("ok"):
                metric["code_json_parse_ok"] += 1
            else:
                metric["code_json_parse_fail"] += 1

            sandbox = run_sandbox(code) if parsed.get("ok") else {"status": "error", "error": parsed.get("error", "parse failed")}
            sandbox_ok = sandbox.get("status") == "ok"
            if sandbox_ok:
                metric["sandbox_ok"] += 1
            else:
                metric["sandbox_fail"] += 1

            pred_result = sandbox.get("result", {}) if isinstance(sandbox.get("result"), dict) else {}
            pred_answer = pred_result.get("answer", "")
            pred_unit = pred_result.get("unit", "")
            pred_kind = pred_result.get("kind", "")

            eq = rule_based_equivalence(pred_answer, pred_unit, ref_answer, ref_unit)
            if eq.get("equivalent") is True:
                metric["answer_rule_correct"] += 1
            elif eq.get("equivalent") is False:
                metric["answer_rule_wrong"] += 1
            else:
                metric["answer_rule_unknown"] += 1

            pinfo, rinfo = unit_info(pred_unit), unit_info(ref_unit)
            if pinfo and rinfo:
                unit_dim_den += 1
                if pinfo[0] == rinfo[0]:
                    unit_dim_num += 1

            ref_kind = row.get("ref_kind", "")
            nk_pred = normalize_text_answer(pred_kind)
            nk_ref = normalize_text_answer(ref_kind)
            if nk_pred and nk_ref:
                kind_match_den += 1
                if nk_pred == nk_ref or nk_pred in nk_ref or nk_ref in nk_pred:
                    kind_match_num += 1

            rec = {
                "problem_id": pid,
                "question": question,
                "prompt": row.get("prompt", ""),
                "completion_text": completion,
                "parsed_ok": bool(parsed.get("ok")),
                "parse_error": parsed.get("error", ""),
                "generated_code": code,
                "sandbox_status": sandbox.get("status"),
                "sandbox_error": sandbox.get("error", ""),
                "sandbox_result": pred_result,
                "pred_answer": pred_answer,
                "pred_unit": pred_unit,
                "pred_kind": pred_kind,
                "ref_answer": ref_answer,
                "ref_unit": ref_unit,
                "ref_kind": ref_kind,
                "equivalence": eq,
                "ref_runner_result": ref_runner,
                "ref_explainer": ref_explainer,
            }
            results.append(rec)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def _rate(num, den):
    return float(num) / float(den) if den else 0.0

summary = {
    "n_eval": n_eval,
    "code_json_parse_rate": _rate(metric["code_json_parse_ok"], n_eval),
    "sandbox_success_rate": _rate(metric["sandbox_ok"], n_eval),
    "answer_rule_accuracy_all": _rate(metric["answer_rule_correct"], n_eval),
    "answer_rule_accuracy_when_sandbox_ok": _rate(
        sum(1 for r in results if r["sandbox_status"] == "ok" and r["equivalence"].get("equivalent") is True),
        metric["sandbox_ok"],
    ),
    "answer_rule_wrong_rate": _rate(metric["answer_rule_wrong"], n_eval),
    "answer_rule_unknown_rate": _rate(metric["answer_rule_unknown"], n_eval),
    "unit_dimension_match_rate": _rate(unit_dim_num, unit_dim_den),
    "unit_dimension_eval_count": unit_dim_den,
    "kind_match_rate": _rate(kind_match_num, kind_match_den),
    "kind_eval_count": kind_match_den,
    "avg_generated_code_chars": _rate(code_chars_total, n_eval),
    "results_path": str(POST_GRPO_RESULTS_PATH),
}

summary_path = POST_GRPO_RESULTS_PATH.with_suffix(".summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nPost-GRPO validation summary")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

print("\nStatus counts")
print(json.dumps(metric, indent=2))
print(f"\nSaved JSONL:   {POST_GRPO_RESULTS_PATH}")
print(f"Saved summary: {summary_path}")


In [ ]:
# ========================================================
# STEP 13: Push final RL adapter and tokenizer to Hugging Face Hub
# ========================================================
if args.push_to_hub:
    print(f"Pushing final RL adapter to: {args.hub_model_id}")
    trainer.push_to_hub()
    tokenizer.push_to_hub(args.hub_model_id, private=args.hub_private_repo)
    print("Push complete.")
else:
    print("push_to_hub=False; skipping final Hub push.")


In [ ]:
# ========================================================
# STEP 14: Optional merge RL LoRA into a full model repo
# ========================================================
# Set MERGE_AND_PUSH=True after you are happy with the RL adapter.
MERGE_AND_PUSH = True
MERGED_HUB_ID = args.hub_model_id.replace("-lora", "-merged")

if MERGE_AND_PUSH:
    print(f"Merging RL LoRA and pushing full model to {MERGED_HUB_ID}")
    merged_model = model.merge_and_unload() if hasattr(model, "merge_and_unload") else model
    merged_model.push_to_hub(MERGED_HUB_ID, private=args.hub_private_repo)
    tokenizer.push_to_hub(MERGED_HUB_ID, private=args.hub_private_repo)
    print("Merged model pushed.")
else:
    print("Skipping merge. Set MERGE_AND_PUSH=True to push a merged full model.")
